In [2]:
!pip install -q -U google-genai

In [22]:
from google import genai
from google.colab import userdata
from google.genai import types
import json
from pydantic import BaseModel
from typing import Optional
client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))

MODEL="gemini-3.5-flash-lite"



In [23]:
prompt="""
extract internship details from this linkdin post.
return only json object that contains all 6keys(role,company,duration_month,stipend,location,last_date) without skipping any key
if any value is not stated in job post ,use null value,dont skip any key
linkdin post:Hiring frontend developer intern at rozorpay,bangaluru
            6month duration  apply by 21august
            For every key:
- Extract the value only if it is explicitly stated in the post.
- If a value is not explicitly stated, return null.
- Never infer, guess, or use a value belonging to another field.
- For example, "21 August" means last_date = "21 August", NOT stipend = 21."""

#way 1 ask llm politely
response1=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.8,
        max_output_tokens=1500,
        system_instruction="You are usefull assistant",
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )

)
print(response1.text)
print("\n\n\n")
#way 2--MiME type response formating
response2=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.8,
        max_output_tokens=1500,
        response_mime_type='application/json',
        system_instruction="You are usefull assistant",
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )
)
data_2=json.loads(response2.text)
print(json.dumps(data_2,indent=2))



#way3: response formating with database schema

class internship(BaseModel):
  role:Optional[str]
  company:Optional[str]
  duration_month:Optional[int]
  stipend:Optional[float]
  location:Optional[str]
  last_date:Optional[str]


response3=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.8,
        max_output_tokens=1500,
        response_mime_type='application/json',
        system_instruction="You are usefull assistant",
        thinking_config=types.ThinkingConfig(thinking_level='low'),
        response_schema=internship
    )
)

data_3=json.loads(response3.text)
expected_keys=['role','company','duration_month','stipend','location','last_date']
for key in expected_keys:
  data_3.setdefault(key,None)


print(json.dumps(data_3,indent=4))


```json
{
  "role": "frontend developer intern",
  "company": "rozorpay",
  "duration_month": "6",
  "stipend": null,
  "location": "bangaluru",
  "last_date": "21 august"
}
```




{
  "role": "frontend developer intern",
  "company": "rozorpay",
  "duration_month": 6,
  "stipend": null,
  "location": "bangaluru",
  "last_date": "21 august"
}
{
    "role": "frontend developer intern",
    "company": "rozorpay",
    "duration_month": 6,
    "stipend": null,
    "location": "bangaluru",
    "last_date": "21 august"
}
